# 01 — Image Exploration

Visual sanity check for the binary-to-grayscale pipeline.  
Each malware family should show a visually distinct texture pattern.

**Run after** `image_gen.py` has populated `data/processed/<family>/`.

In [ ]:
import sys
sys.path.insert(0, '../src')

from pathlib import Path
import random
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from PIL import Image

PROCESSED_DIR = Path('../data/processed')
SAMPLES_PER_FAMILY = 4
random.seed(42)

In [ ]:
family_dirs = sorted(d for d in PROCESSED_DIR.iterdir() if d.is_dir())
print(f"Found {len(family_dirs)} families:")
for d in family_dirs:
    n = len(list(d.glob('*.png')))
    print(f"  {d.name:<25} {n} images")

In [ ]:
fig, axes = plt.subplots(
    len(family_dirs), SAMPLES_PER_FAMILY,
    figsize=(SAMPLES_PER_FAMILY * 3, len(family_dirs) * 3)
)
if len(family_dirs) == 1:
    axes = [axes]

for row, family_dir in enumerate(family_dirs):
    all_imgs = list(family_dir.glob('*.png'))
    samples  = random.sample(all_imgs, min(SAMPLES_PER_FAMILY, len(all_imgs)))

    for col in range(SAMPLES_PER_FAMILY):
        ax = axes[row][col]
        if col < len(samples):
            img = np.array(Image.open(samples[col]).convert('L'))
            ax.imshow(img, cmap='gray', vmin=0, vmax=255)
            ax.set_title(samples[col].stem[:20], fontsize=7)
        else:
            ax.axis('off')

        if col == 0:
            ax.set_ylabel(family_dir.name, fontsize=9, rotation=0, labelpad=80, va='center')
        ax.set_xticks([])
        ax.set_yticks([])

plt.suptitle('Malware family grayscale visualizations', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('../outputs/family_samples.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved to outputs/family_samples.png')

## Per-family byte distribution

Histogram of pixel intensities for one sample per family.  
Obfuscator.ACY samples should appear very flat / uniform — that is the obfuscation effect.

In [ ]:
n = len(family_dirs)
cols = 4
rows = (n + cols - 1) // cols
fig, axes = plt.subplots(rows, cols, figsize=(cols * 4, rows * 3))
axes = axes.flatten()

for i, family_dir in enumerate(family_dirs):
    imgs = list(family_dir.glob('*.png'))
    if not imgs:
        continue
    img = np.array(Image.open(imgs[0]).convert('L'))
    axes[i].hist(img.ravel(), bins=64, range=(0, 255), color='steelblue', edgecolor='none')
    axes[i].set_title(family_dir.name, fontsize=9)
    axes[i].set_xlabel('Pixel intensity')
    axes[i].set_ylabel('Count')

for j in range(i + 1, len(axes)):
    axes[j].axis('off')

plt.suptitle('Byte-intensity histograms (one sample per family)', fontsize=12)
plt.tight_layout()
plt.savefig('../outputs/byte_histograms.png', dpi=150, bbox_inches='tight')
plt.show()